In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report

# ========== 資料讀取與合併 ==========
def time_to_seconds(tstr):
    if pd.isnull(tstr): return 0
    parts = str(tstr).split(':')
    if len(parts) == 3:
        h, m, s = int(parts[0]), int(parts[1]), float(parts[2])
        return h * 3600 + m * 60 + s
    elif len(parts) == 2:
        m, s = int(parts[0]), float(parts[1])
        return m * 60 + s
    else:
        try:
            return float(parts[0])
        except:
            return 0

# 1. 讀檔
winner = pd.read_csv('./data/winners.csv')
drivers = pd.read_csv('./data/drivers_updated.csv')
teams = pd.read_csv('./data/teams_updated.csv')
laps = pd.read_csv('./data/fastest_laps_updated.csv')

# 2. 新增 year 欄（取自 Date）
winner['year'] = pd.to_datetime(winner['Date']).dt.year

# ====== 備份原始年份與分站名（for 預測展示） ======
winner['year_raw'] = winner['year']
winner['Grand Prix raw'] = winner['Grand Prix']

# 3. 合併 drivers
df = winner.merge(
    drivers[['Driver', 'Car', 'year', 'Nationality', 'PTS']],
    left_on=['Winner', 'Car', 'year'],
    right_on=['Driver', 'Car', 'year'],
    how='left',
    suffixes=('', '_driver')
)

# 4. 合併 fastest_laps
df = df.merge(
    laps[['Grand Prix', 'Driver', 'Car', 'year', 'Time']],
    left_on=['Grand Prix', 'Winner', 'Car', 'year'],
    right_on=['Grand Prix', 'Driver', 'Car', 'year'],
    how='left',
    suffixes=('', '_lap')
)

# 5. 合併 teams
df = df.merge(
    teams[['Team', 'PTS', 'year']],
    left_on=['Car', 'year'],
    right_on=['Team', 'year'],
    how='left',
    suffixes=('', '_team')
)

# 6. 時間轉換
df['RaceTime_sec'] = df['Time'].apply(time_to_seconds)
df['FastestLap_sec'] = df['Time_lap'].apply(time_to_seconds)

# ====== 將原始 year/raw 分站資訊對齊帶入 df ======
df['year_raw'] = winner['year_raw']
df['Grand Prix raw'] = winner['Grand Prix raw']

# ========== 特徵前處理 ==========
cat_cols = ['Winner', 'Car', 'Grand Prix', 'Nationality', 'Team']
num_cols = [
    'Laps',
    'PTS',        # drivers_updated.csv 的積分
    'PTS_team',   # teams_updated.csv 的積分
    'RaceTime_sec',
    'FastestLap_sec',
    'year'
]

# 填補數值缺漏
df['PTS_team'] = df['PTS_team'].fillna(0)
df[num_cols] = df[num_cols].fillna(0)

# ========== 過濾只出現一次的冠軍 ==========
value_counts = df['Winner'].value_counts()
valid_drivers = value_counts[value_counts >= 2].index
df = df[df['Winner'].isin(valid_drivers)]

print("過濾前總樣本數：", winner.shape[0])
print("只出現過一次的車手數量：", sum(value_counts == 1))
print("保留後樣本數：", df.shape[0])

# ========== 重新 LabelEncode ==========
le_winner = LabelEncoder()
df['Winner'] = le_winner.fit_transform(df['Winner'].astype(str))

encoders = {'Winner': le_winner}
for col in cat_cols:
    if col != 'Winner':
        le2 = LabelEncoder()
        df[col] = le2.fit_transform(df[col].astype(str))
        encoders[col] = le2

# 數值標準化
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# ========== 資料準備 ==========
X_cat = df[cat_cols].values
X_num = df[num_cols].values
y = df['Winner'].values
num_classes = len(np.unique(y))
print('y範圍:', y.min(), y.max(), 'num_classes:', num_classes)

# ====== 備份原始 year/grandprix 對應到分割前的 df index（以利顯示） ======
df = df.reset_index(drop=True)
df['orig_index'] = df.index  # 加一欄 index，等會方便取 row

# ========== 分割訓練/測試集 ==========
X_cat_train, X_cat_test, X_num_train, X_num_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_cat, X_num, y, df['orig_index'].values, test_size=0.2, random_state=42, stratify=y
)

# ========== PyTorch Dataset ==========
class F1RaceSet(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X_cat[idx], self.X_num[idx], self.y[idx]

# ========== Model ==========
class F1DNN(nn.Module):
    def __init__(self, cat_dims, num_num_features, embedding_dim=8, hidden_dim=128, num_classes=None):
        super().__init__()
        self.emb_layers = nn.ModuleList([
            nn.Embedding(cat_dim, embedding_dim) for cat_dim in cat_dims
        ])
        input_dim = embedding_dim * len(cat_dims) + num_num_features
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.emb_layers)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.mlp(x)

# ========== DataLoader ==========
batch_size = 128
trainset = F1RaceSet(X_cat_train, X_num_train, y_train)
testset = F1RaceSet(X_cat_test, X_num_test, y_test)
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(testset, batch_size=batch_size)

# ========== 訓練流程 ==========
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cat_dims = [int(df[col].max() + 1) for col in cat_cols]

model = F1DNN(cat_dims, len(num_cols), embedding_dim=8, hidden_dim=128, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_cat_batch, X_num_batch, y_batch in train_loader:
        X_cat_batch, X_num_batch, y_batch = X_cat_batch.to(device), X_num_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_cat_batch, X_num_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
    avg_loss = total_loss / len(trainset)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

# ========== 評估 ==========

model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for X_cat_batch, X_num_batch, y_batch in test_loader:
        X_cat_batch, X_num_batch = X_cat_batch.to(device), X_num_batch.to(device)
        logits = model(X_cat_batch, X_num_batch)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(y_batch.numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

# ========== 修正版：以車手名字顯示分類報告 ==========
unique_y = np.unique(all_labels)
target_names = [f"{idx}: {name}" for idx, name in zip(unique_y, le_winner.inverse_transform(unique_y))]
print(classification_report(
    all_labels, all_preds,
    labels=unique_y,
    target_names=target_names,
    zero_division=0
))

# ========== 儲存模型 ==========
torch.save(model.state_dict(), 'f1_dnn_embedding.pth')

# ========== 隨機抽10筆測試集預測結果 ==========
winner_decoder = encoders['Winner']

test_df = df.iloc[idx_test].reset_index(drop=True)

np.random.seed(42)
rand_idx = np.random.choice(len(test_df), size=10, replace=False)

for i in rand_idx:
    row = test_df.iloc[i]
    year = int(row['year_raw'])
    grand_prix = row['Grand Prix raw']
    true_winner = winner_decoder.inverse_transform([all_labels[i]])[0]
    pred_winner = winner_decoder.inverse_transform([all_preds[i]])[0]
    print(f"{year} {grand_prix}\nPredicted Winner: {pred_winner} , True Winner: {true_winner}\n")




過濾前總樣本數： 1110
只出現過一次的車手數量： 36
保留後樣本數： 1074
y範圍: 0 78 num_classes: 79
Epoch 1/30, Loss: 4.3444
Epoch 2/30, Loss: 3.7102
Epoch 3/30, Loss: 3.1566
Epoch 4/30, Loss: 2.7714
Epoch 5/30, Loss: 2.4817
Epoch 6/30, Loss: 2.2333
Epoch 7/30, Loss: 2.0052
Epoch 8/30, Loss: 1.7899
Epoch 9/30, Loss: 1.6386
Epoch 10/30, Loss: 1.4906
Epoch 11/30, Loss: 1.3414
Epoch 12/30, Loss: 1.2467
Epoch 13/30, Loss: 1.1209
Epoch 14/30, Loss: 1.0132
Epoch 15/30, Loss: 0.9487
Epoch 16/30, Loss: 0.8533
Epoch 17/30, Loss: 0.7845
Epoch 18/30, Loss: 0.7103
Epoch 19/30, Loss: 0.6645
Epoch 20/30, Loss: 0.6073
Epoch 21/30, Loss: 0.5711
Epoch 22/30, Loss: 0.5028
Epoch 23/30, Loss: 0.4671
Epoch 24/30, Loss: 0.4232
Epoch 25/30, Loss: 0.4083
Epoch 26/30, Loss: 0.3916
Epoch 27/30, Loss: 0.3505
Epoch 28/30, Loss: 0.3275
Epoch 29/30, Loss: 0.2993
Epoch 30/30, Loss: 0.2806
                             precision    recall  f1-score   support

           0: Alain  Prost        0.91      1.00      0.95        10
             1: Alan 

In [5]:
import gradio as gr

winners_df = winner  # winner = pd.read_csv('./data/winners.csv')
drivers_df = pd.read_csv('./data/drivers_updated.csv')

# 取得年份與分站選單
year_choices = sorted([int(x) for x in winners_df['year'].dropna().unique()])
grand_prix_choices = sorted([str(x) for x in winners_df['Grand Prix'].dropna().unique()])

def get_driver_combos(year, grand_prix, future_mode=False, future_max_year=None):
    if future_mode:
        # 未來年份只納入近兩年出賽該站的車手
        assert future_max_year is not None
        recent_years = [future_max_year, future_max_year-2]
        df = winners_df[
            (winners_df['Grand Prix'] == grand_prix) &
            (winners_df['year'].isin(recent_years))
        ]
    else:
        # 過去年份 → 只用該年該站有出賽車手
        df = winners_df[(winners_df['year'] == int(year)) & (winners_df['Grand Prix'] == grand_prix)]
    combos = []
    for _, row in df.iterrows():
        driver = str(row['Winner'])
        car = str(row['Car'])
        team = str(row['Car'])  # 或 Team 欄
        nationality = drivers_df[
            (drivers_df['Driver'] == driver) & (drivers_df['year'] == int(row['year']))
        ]['Nationality']
        nationality = nationality.values[0] if not nationality.empty else "UNK"
        combos.append(dict(driver=driver, car=car, team=team, nationality=nationality))
    # 過濾唯一組合
    unique_combos = {}
    for c in combos:
        k = (c['driver'], c['car'], c['team'], c['nationality'])
        unique_combos[k] = c
    return list(unique_combos.values())

def predict_winner(year, grand_prix):
    # 判斷是否為未來年份
    max_data_year = max(year_choices)
    year = int(year)
    is_future = year > max_data_year

    combos = get_driver_combos(
        year, grand_prix, future_mode=is_future, future_max_year=max_data_year
    )
    if not combos:
        return "查無該年該場比賽資料或近兩年無車手出賽，無法預測。"
    probs = []
    for combo in combos:
        cat_inputs = [
            encoders['Winner'].transform([combo['driver']])[0] if combo['driver'] in encoders['Winner'].classes_ else 0,
            encoders['Car'].transform([combo['car']])[0] if combo['car'] in encoders['Car'].classes_ else 0,
            encoders['Grand Prix'].transform([grand_prix])[0] if grand_prix in encoders['Grand Prix'].classes_ else 0,
            encoders['Nationality'].transform([combo['nationality']])[0] if combo['nationality'] in encoders['Nationality'].classes_ else 0,
            encoders['Team'].transform([combo['team']])[0] if combo['team'] in encoders['Team'].classes_ else 0
        ]
        cat_inputs = np.array(cat_inputs).reshape(1, -1)
        num_inputs = [0] * len(num_cols)
        if 'year' in num_cols:
            idx = num_cols.index('year')
            num_inputs[idx] = year
        num_inputs = scaler.transform([num_inputs])
        cat_tensor = torch.tensor(cat_inputs, dtype=torch.long).to(device)
        num_tensor = torch.tensor(num_inputs, dtype=torch.float32).to(device)
        model.eval()
        with torch.no_grad():
            logits = model(cat_tensor, num_tensor)
            prob = torch.softmax(logits, dim=1).cpu().numpy()[0]
            probs.append((prob[cat_inputs[0][0]], combo['driver']))
    probs.sort(reverse=True)
    best_driver = probs[0][1]
    prob_str = "\n".join([f"{d}: {p*100:.2f}%" for p, d in probs[:5]])
    year_desc = "未來預測" if is_future else f"{year}年"
    return f"{year_desc} {grand_prix} 預測最有可能奪冠：{best_driver}\n\n[依機率排序前5名]\n{prob_str}"

with gr.Blocks() as demo:
    year_in = gr.Dropdown(choices=year_choices + [max(year_choices)+1], value=year_choices[-1], label="Year (年份，可選未來)")
    grand_prix_in = gr.Dropdown(choices=grand_prix_choices, value=grand_prix_choices[0], label="Grand Prix (場地/分站)")
    out_box = gr.Textbox(label="預測結果")
    btn = gr.Button("預測")
    btn.click(
        predict_winner,
        inputs=[year_in, grand_prix_in],
        outputs=out_box
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
